# Structured Authority Decisions for Tool Execution

This cookbook shows how to intercept tool execution with a structured
authority policy that returns machine-parseable denials, distinguishes
recoverable from permanent rejections, and requires no model, API key,
or Docker dependency.

Unlike the [basic user approval pattern](tool-use-with-intervention.ipynb)
which uses a yes/no prompt, this recipe separates three policy states:

- `AIPOU_AUTHORITY_REQUIRED`: authority is missing but recoverable
- `AIPOU_AUTHORITY_ACCEPTED`: authority is present, execution proceeds
- `AIPOU_ACTION_FORBIDDEN`: action is permanently denied, no retry

An agent loop can branch on the structured denial code without parsing
natural language, and the `canRequestAuthority` flag tells the loop whether
to request authority and retry or to stop immediately.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any, Callable

from autogen_core import AgentId, DefaultInterventionHandler, FunctionCall, MessageContext
from autogen_core.tool_agent import ToolException

## Define the authority decision

`AuthorityDecision` is the structured result of a policy check. The
`can_request_authority` flag distinguishes recoverable denials (the
agent may request authority and retry) from permanent denials (the
agent must not retry).

In [ ]:
@dataclass(frozen=True)
class AuthorityDecision:
    allowed: bool
    code: str
    message: str
    action_ref: str
    can_request_authority: bool

## Define the authority policy

`InMemoryAuthorityPolicy` is a synthetic policy store for the fixture.
It maintains a set of permanently forbidden tools and a map of granted
authority receipts. The `decide` method returns an `AuthorityDecision`
for a given `FunctionCall`.

In [ ]:
class InMemoryAuthorityPolicy:
    """Synthetic policy store for a reproducible fixture."""

    def __init__(self, permanently_forbidden_tools: set[str] | None = None) -> None:
        self._permanently_forbidden_tools = permanently_forbidden_tools or set()
        self._authority_by_action: dict[str, str] = {}

    @staticmethod
    def action_ref(call: FunctionCall) -> str:
        return f"autogen:tool:{call.name}:{call.id}"

    def grant(self, action_ref: str, authority_receipt_id: str) -> None:
        self._authority_by_action[action_ref] = authority_receipt_id

    def decide(self, call: FunctionCall) -> AuthorityDecision:
        action_ref = self.action_ref(call)
        if call.name in self._permanently_forbidden_tools:
            return AuthorityDecision(
                allowed=False,
                code="AIPOU_ACTION_FORBIDDEN",
                message="Tool action is permanently forbidden by orchestrator policy.",
                action_ref=action_ref,
                can_request_authority=False,
            )
        if action_ref not in self._authority_by_action:
            return AuthorityDecision(
                allowed=False,
                code="AIPOU_AUTHORITY_REQUIRED",
                message="Matching pre-action authority is required before tool execution.",
                action_ref=action_ref,
                can_request_authority=True,
            )
        return AuthorityDecision(
            allowed=True,
            code="AIPOU_AUTHORITY_ACCEPTED",
            message="Matching pre-action authority is present.",
            action_ref=action_ref,
            can_request_authority=False,
        )

## Define the intervention handler

`AuthorityInterventionHandler` intercepts `FunctionCall` messages before
they reach the tool agent. On denial, it raises `ToolException` with a
structured JSON body that an agent loop can parse without natural language
processing. On approval, it returns the original `FunctionCall` unchanged.

In [ ]:
def parse_denial(exception: ToolException) -> dict[str, Any]:
    return json.loads(exception.content)


class AuthorityInterventionHandler(DefaultInterventionHandler):
    """Intercept AutoGen FunctionCall messages before ToolAgent execution."""

    def __init__(self, policy: Callable[[FunctionCall], AuthorityDecision]) -> None:
        self._policy = policy

    async def on_send(
        self,
        message: Any,
        *,
        message_context: MessageContext,
        recipient: AgentId,
    ) -> Any:
        if not isinstance(message, FunctionCall):
            return message

        decision = self._policy(message)
        if decision.allowed:
            return message

        denial = {
            "code": decision.code,
            "message": decision.message,
            "actionRef": decision.action_ref,
            "canRequestAuthority": decision.can_request_authority,
            "enforcementPointKind": "orchestrator_policy",
        }
        raise ToolException(content=json.dumps(denial, sort_keys=True))

## Test: missing authority returns a recoverable denial

When no authority receipt exists for the action, the handler raises
`ToolException` with `AIPOU_AUTHORITY_REQUIRED` and `canRequestAuthority`
set to `true`. An agent loop receiving this denial knows it may request
authority and retry once.

In [ ]:
from autogen_core import CancellationToken

policy = InMemoryAuthorityPolicy()
handler = AuthorityInterventionHandler(policy.decide)
call = FunctionCall(id="call-1", name="write_file", arguments='{"path":"fixture.txt"}')

try:
    await handler.on_send(
        call,
        message_context=MessageContext(
            sender=None,
            topic_id=None,
            is_rpc=True,
            cancellation_token=CancellationToken(),
            message_id="test-missing",
        ),
        recipient=AgentId("tool_executor_agent", "default"),
    )
except ToolException as error:
    denial = parse_denial(error)
    print(json.dumps(denial, indent=2))
    assert denial["code"] == "AIPOU_AUTHORITY_REQUIRED"
    assert denial["canRequestAuthority"] is True
    print("\nPass: missing authority is recoverable")

## Test: granted authority allows the original call through

After authority is granted (the receipt is stored in the policy), the
handler returns the original `FunctionCall` unchanged so execution
proceeds normally.

In [ ]:
policy.grant(policy.action_ref(call), "0x" + "11" * 32)

result = await handler.on_send(
    call,
    message_context=MessageContext(
        sender=None,
        topic_id=None,
        is_rpc=True,
        cancellation_token=CancellationToken(),
        message_id="test-granted",
    ),
    recipient=AgentId("tool_executor_agent", "default"),
)

assert result is call
print("Pass: granted authority allows execution")

## Test: permanently forbidden tools cannot be executed even with authority

A tool in the `permanently_forbidden_tools` set is denied regardless of
whether authority was granted. The denial code is `AIPOU_ACTION_FORBIDDEN`
with `canRequestAuthority` set to `false`. An agent loop receiving this
denial knows it must not retry.

In [ ]:
forbidden_call = FunctionCall(id="call-3", name="delete_production", arguments="{}")
policy.grant(policy.action_ref(forbidden_call), "0x" + "22" * 32)

try:
    await handler.on_send(
        forbidden_call,
        message_context=MessageContext(
            sender=None,
            topic_id=None,
            is_rpc=True,
            cancellation_token=CancellationToken(),
            message_id="test-forbidden",
        ),
        recipient=AgentId("tool_executor_agent", "default"),
    )
except ToolException as error:
    denial = parse_denial(error)
    print(json.dumps(denial, indent=2))
    assert denial["code"] == "AIPOU_ACTION_FORBIDDEN"
    assert denial["canRequestAuthority"] is False
    print("\nPass: permanent denial is not retryable")

## Test: non-tool messages pass through unchanged

Messages that are not `FunctionCall` instances bypass the policy check
entirely. The handler returns them unchanged so regular agent-to-agent
communication is not affected.

In [ ]:
regular_message = {"content": "hello from agent"}

result = await handler.on_send(
    regular_message,
    message_context=MessageContext(
        sender=None,
        topic_id=None,
        is_rpc=True,
        cancellation_token=CancellationToken(),
        message_id="test-passthrough",
    ),
    recipient=AgentId("tool_executor_agent", "default"),
)

assert result is regular_message
print("Pass: non-tool messages bypass the policy gate")

## Summary

This pattern extends the basic [user approval intervention](tool-use-with-intervention.ipynb)
with three properties that make it suitable for autonomous agent loops:

| Property | Basic approval | Structured authority |
|---|---|---|
| Denial format | Natural language | JSON with `code` and `canRequestAuthority` |
| Recoverable vs permanent | Single path (approve or drop) | Separate `AUTHORITY_REQUIRED` (retry once) vs `ACTION_FORBIDDEN` (no retry) |
| Dependencies | Requires model, API key, Docker | None (runs in seconds) |

The structured denial lets an agent loop branch on `code` without parsing
natural language. The `canRequestAuthority` flag prevents infinite retry
loops on permanently forbidden actions while allowing one retry for
recoverable ones. The zero-dependency fixture makes the pattern easy to
reproduce and test in CI without external services.